# Sigmoid, tanh and ReLU on MNIST

The same 784-16-16-10 network, trained three times with a different activation each time,
learns at three different speeds, and once it is eight hidden layers deep one of the three
never learns at all. Here you write the activations and their slopes, train the network with
each one, measure how much of the gradient survives the trip back through a deep network,
and repair the deep ReLU network with a better initialization. Open it in Colab or Jupyter:
the first run downloads MNIST, about 12 MB, and the checks need all 60,000 training images.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It loads the libraries and the data every part uses.

In [ ]:
import torch
from torch import nn
from torchvision import datasets

train = datasets.MNIST("data", train=True, download=True)
test = datasets.MNIST("data", train=False, download=True)
X_train = train.data.reshape(-1, 784).float() / 255     # (60000, 784), pixels in [0, 1]
y_train = train.targets
X_test = test.data.reshape(-1, 784).float() / 255       # (10000, 784)
y_test = test.targets

ACTIVATIONS = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}
loss_fn = nn.CrossEntropyLoss()


def accuracy(model):
    """Test-set accuracy of a model that maps (n, 784) images to (n, 10) scores."""
    with torch.no_grad():
        return (model(X_test).argmax(dim=1) == y_test).float().mean().item()


print(X_train.shape, X_test.shape)

## Part 1 (coding task): The three activations and their slopes, by hand

Write each activation and its derivative from the formulas, using only `torch.exp`, `torch.abs`, `torch.where`, comparisons and arithmetic (not `torch.sigmoid`, `torch.tanh` or autograd):

| | $f(z)$ | $f'(z)$ |
|---|---|---|
| sigmoid | $\sigma(z) = 1/(1 + e^{-z})$ | $\sigma(z)\,(1 - \sigma(z))$ |
| tanh | $\tanh(z) = 2\sigma(2z) - 1$ | $1 - \tanh^2(z)$ |
| ReLU | $\max(0, z)$ | $1$ if $z > 0$, else $0$ |

They must work for any input, including $z = \pm 100$. The formula $1/(1 + e^{-z})$ computes $e^{100}$ at $z = -100$, and the textbook tanh, $(e^z - e^{-z})/(e^z + e^{-z})$, returns `nan` at $z = 100$. Compute $e = e^{-|z|}$, which is never larger than 1, and use $\sigma(z) = 1/(1 + e)$ for $z \ge 0$ and $\sigma(z) = e/(1 + e)$ for $z < 0$.

In [ ]:
def sigmoid(z):
    # TODO
    raise NotImplementedError


def tanh(z):
    # TODO: use sigmoid
    raise NotImplementedError


def relu(z):
    # TODO
    raise NotImplementedError


def d_sigmoid(z):
    # TODO
    raise NotImplementedError


def d_tanh(z):
    # TODO
    raise NotImplementedError


def d_relu(z):
    # TODO
    raise NotImplementedError

In [ ]:
z = torch.linspace(-8, 8, 161)
for mine, ref in ((sigmoid, torch.sigmoid), (tanh, torch.tanh), (relu, torch.relu)):
    assert torch.allclose(mine(z), ref(z), atol=1e-6), f"{mine.__name__} does not match {ref.__name__}"
extreme = torch.tensor([-100.0, 100.0])
for f, limits in ((sigmoid, [0.0, 1.0]), (tanh, [-1.0, 1.0])):
    out = f(extreme)
    assert not torch.isnan(out).any(), f"{f.__name__} gives nan at z = -100 or 100"
    assert torch.allclose(out, torch.tensor(limits)), f"{f.__name__}(-100) and {f.__name__}(100) should be {limits}"

z = torch.linspace(-6, 6, 121)
for name, mine, ref in (("sigmoid", d_sigmoid, torch.sigmoid), ("tanh", d_tanh, torch.tanh), ("relu", d_relu, torch.relu)):
    zz = z.clone().requires_grad_()
    ref(zz).sum().backward()                          # autograd's slope at every point
    assert torch.allclose(mine(z), zz.grad, atol=1e-6), f"d_{name} does not match autograd"
    print(f"{name:8s} largest slope {mine(z).max().item():.4f}   slope at z = 6: {mine(z)[-1].item():.6f}")
print("Part 1 passed")

## Part 2 (coding task): Build the network and train it with each activation

Write two functions.

- `make_model(name, hidden_layers=2, width=16)` returns an `nn.Sequential`: `hidden_layers` blocks of `nn.Linear` followed by the activation `ACTIVATIONS[name]()`, then a final `nn.Linear(width, 10)`. The first block takes 784 inputs. Build the list of layers in a loop so the depth can change.
- `fit(model, lr=0.1, epochs=5, batch_size=32)` trains with `torch.optim.SGD`. Each epoch, shuffle the 60,000 training images with `torch.randperm`, walk through them in mini-batches of `batch_size`, and take one step per batch: compute the loss, zero the gradients, backpropagate, step. After each epoch append `accuracy(model)` to a list and return the list.

Then train the 784-16-16-10 network once with each activation, calling `torch.manual_seed(0)` right before `make_model` so the three runs start from comparable weights. Store the histories in `results[name]` and print them.

In [ ]:
def make_model(name, hidden_layers=2, width=16):
    # TODO
    raise NotImplementedError


def fit(model, lr=0.1, epochs=5, batch_size=32):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    # TODO
    raise NotImplementedError


results = {}
# TODO: train one 784-16-16-10 network per activation and print its history

In [ ]:
assert sum(p.numel() for p in make_model("relu").parameters()) == 13002, "2 hidden layers of 16 have 13,002 parameters"
assert sum(p.numel() for p in make_model("relu", 8).parameters()) == 14634, "8 hidden layers of 16 have 14,634 parameters"
assert isinstance(make_model("tanh", 3)[5], nn.Tanh) and len(make_model("tanh", 3)) == 7, "Linear, activation, ..., Linear"
assert set(results) == set(ACTIVATIONS) and all(len(h) == 5 for h in results.values()), "five accuracies per activation"
assert all(h[-1] > 0.9 for h in results.values()), "all three should pass 0.9 by epoch 5"
assert results["sigmoid"][0] < min(results["tanh"][0], results["relu"][0]), "after one epoch sigmoid should be behind"
print("Part 2 passed")

## Part 3 (coding task): Watch the gradient vanish, layer by layer

Every layer the gradient passes back through multiplies it by that layer's activation slope. Measure what that does to a deep network.

Write `layer_gradient_sizes(model, X, y)`: zero the model's gradients, compute the loss on `(X, y)`, backpropagate once, and return a list with the mean absolute gradient of the **weight** of every `nn.Linear` layer in the model, from the first layer to the last.

Then, for each activation, build `make_model(name, hidden_layers=8)` right after `torch.manual_seed(0)`, measure it on the first 256 training images **before any training**, and print one row per activation with the nine numbers in scientific notation. Store the lists in `profiles[name]`.

In [ ]:
def layer_gradient_sizes(model, X, y):
    # TODO
    raise NotImplementedError


profiles = {}
# TODO

In [ ]:
assert set(profiles) == set(ACTIVATIONS), "store one list per activation in profiles[name]"
assert all(len(p) == 9 for p in profiles.values()), "8 hidden layers plus the output layer is 9 Linear layers"
ratio = {name: p[0] / p[-1] for name, p in profiles.items()}
print("first layer / last layer:", {name: f"{r:.1e}" for name, r in ratio.items()})
assert profiles["sigmoid"][0] < 1e-6, "eight sigmoid layers should leave almost no gradient at the first layer"
assert ratio["sigmoid"] < 1e-5, "sigmoid's gradient should shrink by more than 100,000 from last to first layer"
assert ratio["tanh"] > 1000 * ratio["sigmoid"] and ratio["relu"] > 1000 * ratio["sigmoid"], "tanh and ReLU should keep far more"
print("Part 3 passed")

## Part 4 (coding task): Train the deep networks, then fix the initialization

Write `train_deep(name, init=None)`: call `torch.manual_seed(0)`, build `make_model(name, hidden_layers=8)`, apply `init(model)` if `init` is given, save a copy of the first layer's weight, train with `fit` for 5 epochs, and return `(history, moved)`, where `moved` is $\lVert W_{\text{after}} - W_{\text{before}}\rVert / \lVert W_{\text{before}}\rVert$ for the first layer's weight. Run it for all three activations.

Then write `he_init(model)`, which re-initializes every `nn.Linear` in the model with `nn.init.kaiming_normal_(weight, nonlinearity="relu")`, sets its bias to zero and returns the model. This draws weights with variance $2/\text{fan-in}$, the value that keeps the size of the signal constant through a ReLU layer. Measure the 8-layer ReLU network's gradient profile with He initialization (seed 0, first 256 images) and train it with `train_deep("relu", he_init)`.

In [ ]:
def train_deep(name, init=None):
    # TODO
    raise NotImplementedError


def he_init(model):
    # TODO
    raise NotImplementedError


deep = {}
# TODO: deep[name] = train_deep(name) for each activation, and print the results

# TODO: he_profile (seed 0, he_init, first 256 images) and he_history from train_deep("relu", he_init)
he_profile = he_history = None

In [ ]:
assert set(deep) == set(ACTIVATIONS), "store train_deep(name) in deep[name] for every activation"
assert he_profile is not None and he_history is not None, "fill in he_profile and he_history"
assert deep["sigmoid"][0][-1] < 0.2, "the deep sigmoid network should be stuck near chance"
assert deep["sigmoid"][1] < 1e-3, "its first layer should barely move"
assert deep["tanh"][0][-1] > 0.85 and deep["relu"][0][-1] > 0.85, "the deep tanh and ReLU networks should train"
assert he_profile[0] / he_profile[-1] > 0.1, "with He initialization the gradient should reach the first layer at full size"
assert he_history[0] > deep["relu"][0][0] + 0.3, "He initialization should train far faster in the first epoch"
print("Part 4 passed")

## Solution

### Part 1: The three activations and their slopes, by hand

In [ ]:
def sigmoid(z):
    e = torch.exp(-torch.abs(z))                      # in (0, 1], never overflows
    return torch.where(z >= 0, 1 / (1 + e), e / (1 + e))


def tanh(z):
    return 2 * sigmoid(2 * z) - 1


def relu(z):
    return torch.where(z > 0, z, torch.zeros_like(z))


def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)


def d_tanh(z):
    return 1 - tanh(z) ** 2


def d_relu(z):
    return (z > 0).float()

**Why it works.** $e^{-|z|}$ is at most 1, so neither branch can overflow, and both branches are the same function (multiply the top and bottom of $1/(1 + e^{-z})$ by $e^{z}$). `torch.where` evaluates both branches, which is why both must be safe for every $z$. The slope of sigmoid is $s(1 - s)$ with $s = \sigma(z)$, and by AM-GM $s(1 - s) \le \tfrac14$, with equality only at $z = 0$. By $z = 6$ it is down to about $0.0025$, and tanh's to about $0.000025$: a neuron sitting there passes almost no gradient back, which is what "saturated" means. ReLU's slope is exactly 1 for every positive input.

### Part 2: Build the network and train it with each activation

In [ ]:
def make_model(name, hidden_layers=2, width=16):
    layers, fan_in = [], 784
    for _ in range(hidden_layers):
        layers += [nn.Linear(fan_in, width), ACTIVATIONS[name]()]
        fan_in = width
    layers.append(nn.Linear(fan_in, 10))
    return nn.Sequential(*layers)


def fit(model, lr=0.1, epochs=5, batch_size=32):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    n = len(X_train)
    for _ in range(epochs):
        order = torch.randperm(n)
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            loss = loss_fn(model(X_train[batch]), y_train[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        history.append(accuracy(model))
    return history


results = {}
for name in ACTIVATIONS:
    torch.manual_seed(0)
    model = make_model(name)
    results[name] = fit(model)
    print(f"{name:8s}", "  ".join(f"{a:.4f}" for a in results[name]))

**Why it works.** Same network, same data, same learning rate. After one epoch tanh and ReLU are at about 0.93 while sigmoid is near 0.79, and sigmoid is still behind after five. Its slope is never above 0.25, so every gradient that passes through a sigmoid layer is cut to a quarter or less, and every step is smaller. Sigmoid catches up if you raise the learning rate. The parameter count is $784 \cdot 16 + 16 + 16 \cdot 16 + 16 + 16 \cdot 10 + 10 = 13{,}002$, and each extra hidden layer adds $16 \cdot 16 + 16 = 272$.

### Part 3: Watch the gradient vanish, layer by layer

In [ ]:
def layer_gradient_sizes(model, X, y):
    model.zero_grad()
    loss_fn(model(X), y).backward()
    return [layer.weight.grad.abs().mean().item()
            for layer in model if isinstance(layer, nn.Linear)]


profiles = {}
for name in ACTIVATIONS:
    torch.manual_seed(0)
    deep_model = make_model(name, hidden_layers=8)
    profiles[name] = layer_gradient_sizes(deep_model, X_train[:256], y_train[:256])
    print(f"{name:8s}", " ".join(f"{g:.1e}" for g in profiles[name]))

**Why it works.** Each sigmoid layer multiplies the gradient by at most 0.25, so eight of them cut it by at least $4^8 = 65{,}536$, and the weight matrices shrink it further. The printed sigmoid row falls by a factor of 6 or more per layer, from about $10^{-2}$ at the output to about $10^{-9}$ at the first layer. Tanh and ReLU have slope 1 near zero or wherever the neuron is on, so they lose far less. They still lose some: PyTorch's default `nn.Linear` initialization draws weights with variance $1/(3\,\text{fan-in})$, smaller than ReLU needs to keep the signal's size constant, and that shrinkage compounds too. Part 4 fixes it.

### Part 4: Train the deep networks, then fix the initialization

In [ ]:
def train_deep(name, init=None):
    torch.manual_seed(0)
    model = make_model(name, hidden_layers=8)
    if init is not None:
        init(model)
    before = model[0].weight.detach().clone()
    history = fit(model)
    moved = ((model[0].weight.detach() - before).norm() / before.norm()).item()
    return history, moved


def he_init(model):
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    return model


deep = {}
for name in ACTIVATIONS:
    deep[name] = train_deep(name)
    history, moved = deep[name]
    print(f"{name:8s}", "  ".join(f"{a:.4f}" for a in history), f"  first layer moved {moved:.1e}")

torch.manual_seed(0)
he_profile = layer_gradient_sizes(he_init(make_model("relu", hidden_layers=8)), X_train[:256], y_train[:256])
print("ReLU + He init gradients:", " ".join(f"{g:.1e}" for g in he_profile))
he_history, _ = train_deep("relu", he_init)
print("ReLU + He init accuracy: ", "  ".join(f"{a:.4f}" for a in he_history))

**Why it works.** The deep sigmoid network's first layer moves by about $5 \times 10^{-5}$ of its size in five epochs: weights that receive no gradient do not change, so the network is stuck at chance, about 0.11, which is the share of the most common digit. This is the vanishing gradient problem and the main reason sigmoid is no longer used between the layers of deep networks. With He initialization each ReLU layer keeps the variance of its input, since half the units are off on average and the factor 2 compensates. The gradient profile becomes flat, and the deep ReLU network reaches about 0.89 after one epoch instead of about 0.18.